# Семинар 07. Протоколы и duck typing


## Цели

После семинара вы сможете:

- различать номинальную и структурную типизацию;
- описывать интерфейсы с ABC и typing.Protocol;
- понимать границы статической проверки типов во время выполнения.

## Перед началом

Повторите наследование, абстрактные классы и аннотации типов. Для демонстрации типов нужен nb-mypy.


## Полезные ссылки

- [Python: `typing.Protocol` и `runtime_checkable`](https://docs.python.org/3/library/typing.html#typing.Protocol)
- [Спецификация типизации: протоколы](https://typing.python.org/en/latest/spec/protocol.html)
- [mypy: protocols and structural subtyping](https://mypy.readthedocs.io/en/stable/protocols.html)

## Динамическая и статическая проверка типов

В Python тип принадлежит объекту, а имя может последовательно ссылаться на объекты разных типов. Аннотации описывают ожидаемые типы, но интерпретатор обычно не проверяет их автоматически. Статический анализатор читает аннотации до запуска и находит часть несовместимостей заранее.

```python
class Rectangle:
    def __init__(self, width: int, height: int) -> None:
        self._width = width
        self._height = height

    def get_area(self) -> int:
        return self._width * self._height


rectangle = Rectangle(10, "tuesday")  # ошибка статической типизации
print(rectangle.get_area())
```

Этот код запускается: умножение строки на целое число допустимо, поэтому программа печатает повторённое слово вместо площади. Аннотация не изменила поведение во время выполнения, но анализатор может указать на ошибочный аргумент и несоответствие обещанному результату `int`.

## Номинальная и структурная совместимость

| Подход | Когда объект считается совместимым | Инструмент Python |
|---|---|---|
| Номинальный | его класс явно входит в нужную иерархию наследования | обычный базовый класс, `abc.ABC`, `isinstance()` |
| Структурный | объект предоставляет требуемые методы и атрибуты совместимых типов | `typing.Protocol` и статический анализатор |

Обычный duck typing — это поведение программы во время выполнения: код вызывает нужный метод, не проверяя происхождение объекта заранее. `Protocol` формализует такую совместимость для статического анализатора.

Проверки `isinstance()` полезны на внешних границах системы, но не проверяют аннотации всех методов и не заменяют валидацию входных данных или статический анализ.

## Абстрактные базовые классы

`abc.ABC` задаёт номинальный контракт: конкретный класс должен унаследовать ABC и реализовать абстрактные методы, прежде чем его можно будет создать. Это удобно, когда интерфейс принадлежит одной иерархии и вместе с ним нужно передать общую реализацию или состояние.


In [ ]:
from abc import ABC, abstractmethod


class Flyable(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...


class Junkie(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...


class Bird(Flyable):
    def fly(self) -> None:
        print("I'm flying")


class Hippie(Junkie):
    def fly(self) -> None:
        print("I'm flying high")


class FlyableManager:
    def __init__(self) -> None:
        self.flyables: list[Flyable] = []

    def add_flyable(self, flyable: Flyable) -> None:
        self.flyables.append(flyable)
    
    def fly(self) -> None:
        for flyable in self.flyables:
            flyable.fly()


manager = FlyableManager()
manager.add_flyable(Bird())
# Статический анализатор отклонит строку ниже: Hippie не наследует Flyable.
# Во время выполнения явной проверки нет, поэтому вызов сработает.
manager.add_flyable(Hippie())
manager.fly()

В примере `Hippie` реализует метод `fly()`, но принадлежит другой номинальной иерархии через `Junkie`. Статический анализатор отклоняет передачу `Hippie` в `add_flyable()`. Интерпретатор всё же выполняет вызов, потому что аннотация параметра не является автоматической runtime-проверкой.

## Протоколы и duck typing

`Protocol` описывает требуемые методы и атрибуты структурно. Явное наследование не требуется: анализатор считает класс совместимым, если все члены протокола присутствуют и имеют совместимые типы. Так статическая проверка формализует привычный Python-подход «если объект ведёт себя как утка, его можно использовать как утку».


In [ ]:
%load_ext nb_mypy
from typing import Protocol


class Flyable(Protocol):
    def fly(self) -> None:
        ...


class Bird:
    def fly(self) -> None:
        print("I'm flying")


class Hippie:
    def fly(self) -> None:
        print("I'm flying high")


class ParametrizedFlyer:
    def fly(self, speed: int) -> None:
        print(f"I'm flying at {speed} knots")


class FlyableManager:
    def __init__(self) -> None:
        self.flyables: list[Flyable] = []

    def add_flyable(self, flyable: Flyable) -> None:
        self.flyables.append(flyable)
    
    def fly(self) -> None:
        for flyable in self.flyables:
            flyable.fly()


manager = FlyableManager()
manager.add_flyable(Bird())
# Hippie не наследуется от Flyable, но структурно соответствует протоколу.
manager.add_flyable(Hippie())
manager.fly()

# Сигнатура fly несовместима: обязательный аргумент speed отсутствует в протоколе.
# manager.add_flyable(ParametrizedFlyer())
# У int вообще нет метода fly.
# manager.add_flyable(1)


## Проверка протокола во время выполнения

Обычный протокол нельзя передать вторым аргументом в `isinstance()`. Декоратор `@runtime_checkable` разрешает такую проверку, но она проверяет только наличие членов протокола — не типы параметров и результата. Поэтому объект с несовместимой сигнатурой может пройти `isinstance()`, а затем завершиться ошибкой при вызове.


In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class RuntimeFlyable(Protocol):
    def fly(self) -> None:
        ...


print(isinstance(Bird(), RuntimeFlyable))
# Метод fly существует, но его обязательный параметр runtime-проверка не видит.
print(isinstance(ParametrizedFlyer(), RuntimeFlyable))


## Как выбрать инструмент

| Ситуация | Подход |
|---|---|
| Классы образуют одну управляемую иерархию, нужна общая реализация | базовый класс или `ABC` |
| Функции достаточно небольшого набора операций от любых подходящих объектов | узкий `Protocol` |
| Код прост и полностью покрыт тестами, статическая проверка не используется | обычный duck typing |
| Нужно проверить внешние данные во время выполнения | явная валидация данных; одного `Protocol` недостаточно |

Чем уже интерфейс, тем проще подобрать реализацию и тестовую замену. Функции, которой нужен только `fly()`, не стоит передавать тип с десятками несвязанных методов.


## Самопроверка

1. Почему аннотация `height: int` не останавливает передачу строки во время выполнения?
2. Почему `Hippie` несовместим с ABC `Flyable`, хотя у него есть метод `fly()`?
3. Почему тот же `Hippie` совместим с протоколом `Flyable`?
4. Как анализатор обнаруживает несовместимость `ParametrizedFlyer` с протоколом?
5. Почему `@runtime_checkable` нельзя использовать для полной проверки сигнатуры?


## Итоги

- Аннотации помогают анализатору, но обычно не проверяются интерпретатором автоматически.
- ABC задаёт номинальную совместимость через наследование.
- `Protocol` задаёт структурную совместимость через набор членов и их типы.
- Duck typing работает во время выполнения независимо от наличия `Protocol`.
- `@runtime_checkable` проверяет наличие членов, но игнорирует их типовые сигнатуры.


## Задание 1. Круглые скобки (1 балл)

Строка состоит только из `(` и `)`. Определите, является ли она правильной скобочной последовательностью.

```python
def is_valid_parentheses(text: str) -> bool:
    ...
```

**Примеры:** `""`, `"()"` и `"(())()"` валидны; `")("` и `"(()"` невалидны.

**Критерии проверки:** `O(N)` по времени, `O(1)` по дополнительной памяти; промежуточный баланс никогда не становится отрицательным и в конце равен нулю.


## Задание 2. Несколько видов скобок (2 балла)

Строка состоит из символов `()[]{}`. Закрывающая скобка должна соответствовать последней незакрытой скобке.

```python
def is_valid_brackets(text: str) -> bool:
    ...
```

**Примеры:** `"[]()"` и `"{[()]}"` валидны; `"[(])"` и `"{{}"` невалидны.

**Критерии проверки:** `O(N)` по времени и `O(N)` по памяти; решение корректно обрабатывает пустую строку и закрывающую скобку в начале.
